## Seminar 1: Fun with Word Embeddings (3 points)

Today we gonna play with word embeddings: train our own little embeddings, load one from gensim model zoo and use it to visualize text corpora.

This whole thing is gonna happen on top of embedding dataset.

__Requirements:__  `pip install --upgrade nltk gensim bokeh` , but only if you're running locally.

In [3]:
# download the data:
!wget "https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1" -O ./quora.txt
# alternative download link: https://yadi.sk/i/BPQrUu1NaTduEw

--2025-10-16 15:59:22--  https://www.dropbox.com/s/obaitrix9jyu84r/quora.txt?dl=1
Resolving www.dropbox.com (www.dropbox.com)... 162.125.66.18
connected. to www.dropbox.com (www.dropbox.com)|162.125.66.18|:443... 
302 Foundest sent, awaiting response... 
Location: https://www.dropbox.com/scl/fi/p0t2dw6oqs6oxpd6zz534/quora.txt?rlkey=bjupppwua4zmd4elz8octecy9&dl=1 [following]
--2025-10-16 15:59:22--  https://www.dropbox.com/scl/fi/p0t2dw6oqs6oxpd6zz534/quora.txt?rlkey=bjupppwua4zmd4elz8octecy9&dl=1
Reusing existing connection to www.dropbox.com:443.
302 Foundest sent, awaiting response... 
Location: https://ucd8f6335e4bf2bc3aa0a8382746.dl.dropboxusercontent.com/cd/0/inline/CzWqIRT1mbPgRKsU9iMImlkrz9xxMrwy2ingQkZCyrfjBgoIW7LNj0OUmW9naO5yOWgYV-7MDMuXDBwhUiVrxYf-Hh543kIfxAvDi_wpjOibh8Ae7hlZrsX00s9OoFOYaJU/file?dl=1# [following]
--2025-10-16 15:59:23--  https://ucd8f6335e4bf2bc3aa0a8382746.dl.dropboxusercontent.com/cd/0/inline/CzWqIRT1mbPgRKsU9iMImlkrz9xxMrwy2ingQkZCyrfjBgoIW7LNj0OUmW9naO5yO

In [4]:
import numpy as np

with open("./quora.txt", encoding="utf-8") as file:
    data = list(file)

data[50]

"What TV shows or books help you read people's body language?\n"

__Tokenization:__ a typical first step for an NLP task is to split raw data into words.
The text we're working with is in raw format: with all the punctuation and smiles attached to some words, so a simple str.split won't do.

Let's use __`nltk`__ - a library that handles many NLP tasks like tokenization, stemming or part-of-speech tagging.

In [6]:
from nltk.tokenize import WordPunctTokenizer
tokenizer = WordPunctTokenizer()

print(tokenizer.tokenize(data[50]))

['What', 'TV', 'shows', 'or', 'books', 'help', 'you', 'read', 'people', "'", 's', 'body', 'language', '?']


In [7]:
data[50]

"What TV shows or books help you read people's body language?\n"

In [8]:
# TASK: lowercase everything and extract tokens with tokenizer. 
# data_tok should be a list of lists of tokens for each line in data.

data_tok = [tokenizer.tokenize(x.lower()) for x in data]

In [9]:
assert all(isinstance(row, (list, tuple)) for row in data_tok), "please convert each line into a list of tokens (strings)"
assert all(all(isinstance(tok, str) for tok in row) for row in data_tok), "please convert each line into a list of tokens (strings)"
is_latin = lambda tok: all('a' <= x.lower() <= 'z' for x in tok)
assert all(map(lambda l: not is_latin(l) or l.islower(), map(' '.join, data_tok))), "please make sure to lowercase the data"

In [10]:
print([' '.join(row) for row in data_tok[:2]])

["can i get back with my ex even though she is pregnant with another guy ' s baby ?", 'what are some ways to overcome a fast food addiction ?']


In [11]:
len(data_tok)

537272

__Word vectors:__ as the saying goes, there's more than one way to train word embeddings. There's Word2Vec and GloVe with different objective functions. Then there's fasttext that uses character-level models to train word embeddings. 

The choice is huge, so let's start someplace small: __gensim__ is another nlp library that features many vector-based models incuding word2vec.

In [13]:
from gensim.models import Word2Vec
model = Word2Vec(data_tok, 
                 vector_size=32,      # embedding vector size
                 min_count=5,  # consider words that occured at least 5 times
                 window=5).wv  # define context as a 5-word window around the target word

# From gensim docs
# vw: This object essentially contains the mapping between words and embeddings.
# After training, it can be used directly to query those embeddings in various ways.

In [14]:
# now you can get word vectors !
model.get_vector('anything')

array([-1.4903525 , -0.17421657,  0.3247588 ,  2.9432888 ,  1.9103713 ,
        2.9585953 ,  2.20136   , -2.647443  ,  0.84090334,  2.8938575 ,
       -1.2950125 ,  2.7208383 ,  4.363931  ,  1.8238243 ,  3.1740444 ,
       -0.78825545,  0.8385316 , -1.9989653 ,  0.74860424, -3.7729385 ,
       -2.5077922 ,  0.9813209 , -1.6010938 , -1.0680784 ,  1.7891672 ,
       -1.7228273 , -0.78120035,  0.7756592 ,  0.42636687, -0.51190275,
       -0.6466582 ,  0.6648989 ], dtype=float32)

In [15]:
# or query similar words directly. Go play with it!
model.most_similar('bread')

[('rice', 0.951143741607666),
 ('sauce', 0.9316589832305908),
 ('cheese', 0.9184439778327942),
 ('soup', 0.9179419279098511),
 ('potato', 0.9176271557807922),
 ('fruit', 0.9124485850334167),
 ('butter', 0.9120150804519653),
 ('honey', 0.9065386056900024),
 ('chocolate', 0.904394268989563),
 ('chicken', 0.9001390337944031)]

In [16]:
model.most_similar('can')

[('should', 0.828060507774353),
 ('could', 0.8218923211097717),
 ('do', 0.8067538738250732),
 ('will', 0.7899186015129089),
 ('shall', 0.7617586851119995),
 ('would', 0.73606276512146),
 ('cannot', 0.711656928062439),
 ('wanna', 0.6786591410636902),
 ('did', 0.6295720934867859),
 ('want', 0.6235451102256775)]

In [17]:
model.most_similar('house')

[('room', 0.7399705648422241),
 ('hand', 0.7275838255882263),
 ('apartment', 0.6996114253997803),
 ('road', 0.6987422108650208),
 ('door', 0.6853686571121216),
 ('car', 0.6849793791770935),
 ('helmet', 0.6837778687477112),
 ('family', 0.6768374443054199),
 ('lane', 0.6750854253768921),
 ('bedroom', 0.67282634973526)]

In [18]:
model.most_similar('putin')

[('vladimir', 0.871987521648407),
 ('barack', 0.8668938875198364),
 ('michelle', 0.8243853449821472),
 ('hitler', 0.8052379488945007),
 ('abraham', 0.8024550080299377),
 ('republican', 0.8009974360466003),
 ('bernie', 0.800900399684906),
 ('democrat', 0.7992088198661804),
 ('supporter', 0.7966254949569702),
 ('edward', 0.7862356305122375)]

In [19]:
model.most_similar('kate')

[('sandor', 0.9644016027450562),
 ('glenn', 0.9605434536933899),
 ('godse', 0.9546054005622864),
 ('nathuram', 0.9506030082702637),
 ('helena', 0.9484757781028748),
 ('mauritania', 0.9484078288078308),
 ('mayday', 0.9473850727081299),
 ('](-', 0.9471597075462341),
 ('slime', 0.9466716051101685),
 ('ferdinand', 0.9450882077217102)]

In [20]:
model.most_similar('machine')

[('hadoop', 0.7432546019554138),
 ('ai', 0.7147916555404663),
 ('algorithms', 0.7032931447029114),
 ('deep', 0.7013768553733826),
 ('material', 0.6799283027648926),
 ('javascript', 0.6739596128463745),
 ('supervised', 0.6737711429595947),
 ('frame', 0.6479228138923645),
 ('plc', 0.6467396020889282),
 ('github', 0.63900226354599)]

In [21]:
model.most_similar('learning')

[('coding', 0.8263176083564758),
 ('learn', 0.8050399422645569),
 ('programming', 0.7922689318656921),
 ('basics', 0.7580858469009399),
 ('python', 0.7453691363334656),
 ('java', 0.7319720387458801),
 ('javascript', 0.7293636798858643),
 ('beginners', 0.7156808376312256),
 ('php', 0.700063943862915),
 ('studying', 0.6945960521697998)]

In [22]:
model.most_similar('alex')

[('dale', 0.8982027769088745),
 ('springsteen', 0.8844276070594788),
 ('homer', 0.8832859396934509),
 ('altman', 0.8807658553123474),
 ('athena', 0.8761308193206787),
 ('trey', 0.8745012283325195),
 ('dame', 0.8731403946876526),
 ('rr', 0.8691931962966919),
 ('gamble', 0.8652203679084778),
 ('zappa', 0.8649952411651611)]

In [23]:
model.most_similar('1967')

[('kippur', 0.8547655344009399),
 ('45th', 0.847369372844696),
 ('condemning', 0.8458608984947205),
 ('coalition', 0.8381814956665039),
 ('democratically', 0.8375044465065002),
 ('sharif', 0.835577666759491),
 ('kalinga', 0.8316877484321594),
 ('vinay', 0.8292083144187927),
 ('nawaz', 0.8246855139732361),
 ('ratified', 0.8203646540641785)]

In [24]:
model.most_similar('1984')

[('persia', 0.896702229976654),
 ('mahabharat', 0.886179506778717),
 ('sarkar', 0.8861538171768188),
 ('arjun', 0.880294144153595),
 ('christopher', 0.8748216032981873),
 ('mahabharata', 0.870396077632904),
 ('martian', 0.8646414875984192),
 ('william', 0.8577247858047485),
 ('deadpool', 0.8541220426559448),
 ('alfred', 0.8521387577056885)]

In [25]:
model.most_similar('bitcoin')

[('stock', 0.8776436448097229),
 ('market', 0.8065527081489563),
 ('paytm', 0.8036348223686218),
 ('stocks', 0.7864423394203186),
 ('investing', 0.7861326336860657),
 ('paypal', 0.7684581875801086),
 ('payment', 0.7551321387290955),
 ('blockchain', 0.7536362409591675),
 ('bitcoins', 0.753393292427063),
 ('aliexpress', 0.7498653531074524)]

### Using pre-trained model

Took it a while, huh? Now imagine training life-sized (100~300D) word embeddings on gigabytes of text: wikipedia articles or twitter posts. 

Thankfully, nowadays you can get a pre-trained word embedding model in 2 lines of code (no sms required, promise).

After being downloaded for the first time (or if you manually delete it), the model is saved in the `~/gensim_data` or `%USER_PATH%/gensim_data` directory. This can be checked seting the return_path parameter to True.

In [27]:
import gensim.downloader as api
model = api.load('glove-twitter-100')

In [28]:
model.most_similar(positive=["coder", "money"], negative=["brain"])

[('broker', 0.5820155739784241),
 ('bonuses', 0.5424473881721497),
 ('banker', 0.5385112762451172),
 ('designer', 0.5197198390960693),
 ('merchandising', 0.4964233338832855),
 ('treet', 0.49220192432403564),
 ('shopper', 0.4920562207698822),
 ('part-time', 0.4912828207015991),
 ('freelance', 0.4843312203884125),
 ('aupair', 0.4796452522277832)]

### Visualizing word vectors

One way to see if our vectors are any good is to plot them. Thing is, those vectors are in 30D+ space and we humans are more used to 2-3D.

Luckily, we machine learners know about __dimensionality reduction__ methods.

Let's use that to plot 1000 most frequent words

In [30]:
words = model.index_to_key[:1000] 

print(words[::100])

['<user>', '_', 'please', 'apa', 'justin', 'text', 'hari', 'playing', 'once', 'sei']


In [31]:
# for each word, compute it's vector with model
word_vectors = model[words]
word_vectors

array([[ 0.63006 ,  0.65177 ,  0.25545 , ...,  0.55096 ,  0.64706 ,
        -0.6093  ],
       [ 0.18205 , -0.048483,  0.23966 , ..., -0.3358  ,  0.18884 ,
        -0.40786 ],
       [ 1.0674  ,  0.45716 ,  0.51463 , ...,  0.13974 ,  0.76487 ,
        -0.17314 ],
       ...,
       [ 0.11065 , -0.040876,  0.2586  , ..., -0.74773 ,  0.306   ,
         0.37911 ],
       [ 0.30491 , -0.89558 , -0.46538 , ...,  0.73649 ,  1.3842  ,
         0.99976 ],
       [-0.43051 ,  0.56302 ,  0.38305 , ..., -0.3222  ,  0.32782 ,
         0.51519 ]], dtype=float32)

In [32]:
assert isinstance(word_vectors, np.ndarray)
assert word_vectors.shape == (len(words), 100)
assert np.isfinite(word_vectors).all()

#### Linear projection: PCA

The simplest linear dimensionality reduction method is **P**rincipial **C**omponent **A**nalysis.

In geometric terms, PCA tries to find axes along which most of the variance occurs. The "natural" axes, if you wish.

<img src="https://github.com/yandexdataschool/Practical_RL/raw/master/yet_another_week/_resource/pca_fish.png" style="width:30%">


Under the hood, it attempts to decompose object-feature matrix $X$ into two smaller matrices: $W$ and $\hat W$ minimizing _mean squared error_:

$$\|(X W) \hat{W} - X\|^2_2 \to_{W, \hat{W}} \min$$
- $X \in \mathbb{R}^{n \times m}$ - object matrix (**centered**);
- $W \in \mathbb{R}^{m \times d}$ - matrix of direct transformation;
- $\hat{W} \in \mathbb{R}^{d \times m}$ - matrix of reverse transformation;
- $n$ samples, $m$ original dimensions and $d$ target dimensions;



In [34]:
from sklearn.decomposition import PCA

# map word vectors onto 2d plane with PCA. Use good old sklearn api (fit, transform)
# after that, normalize vectors to make sure they have zero mean and unit variance
pca = PCA(n_components=2, whiten=True)
word_vectors_pca = pca.fit_transform(word_vectors)

# and maybe MORE OF YOUR CODE here :)
word_vectors_pca.std()

0.99949944

In [35]:
assert word_vectors_pca.shape == (len(word_vectors), 2), "there must be a 2d vector for each word"
assert max(abs(word_vectors_pca.mean(0))) < 1e-5, "points must be zero-centered"
assert max(abs(1.0 - word_vectors_pca.std(0))) < 1e-2, "points must have unit variance"

#### Let's draw it!

In [37]:
import bokeh.models as bm, bokeh.plotting as pl
from bokeh.io import output_notebook
output_notebook()

def draw_vectors(x, y, radius=10, alpha=0.25, color='blue',
                 width=600, height=400, show=True, **kwargs):
    """ draws an interactive plot for data points with auxilirary info on hover """
    if isinstance(color, str): color = [color] * len(x)
    data_source = bm.ColumnDataSource({ 'x' : x, 'y' : y, 'color': color, **kwargs })

    fig = pl.figure(active_scroll='wheel_zoom', width=width, height=height)
    fig.scatter('x', 'y', size=radius, color='color', alpha=alpha, source=data_source)

    fig.add_tools(bm.HoverTool(tooltips=[(key, "@" + key) for key in kwargs.keys()]))
    if show: pl.show(fig)
    return fig

Loading BokehJS ...

In [38]:
draw_vectors(word_vectors_pca[:, 0], word_vectors_pca[:, 1], token=words)

# hover a mouse over there and see if you can identify the clusters

figure(id='p1004', ...)

### Visualizing neighbors with t-SNE
PCA is nice but it's strictly linear and thus only able to capture coarse high-level structure of the data.

If we instead want to focus on keeping neighboring points near, we could use TSNE, which is itself an embedding method. Here you can read __[more on TSNE](https://distill.pub/2016/misread-tsne/)__.

In [40]:
from sklearn.manifold import TSNE

# map word vectors onto 2d plane with TSNE. hint: don't panic it may take a minute or two to fit.
# normalize them as just lke with pca
tsne = TSNE(
    n_components=2
)

word_tsne = tsne.fit_transform(word_vectors)

In [41]:
draw_vectors(word_tsne[:, 0], word_tsne[:, 1], color='green', token=words)

figure(id='p1055', ...)

### Visualizing phrases

Word embeddings can also be used to represent short phrases. The simplest way is to take __an average__ of vectors for all tokens in the phrase with some weights.

This trick is useful to identify what data are you working with: find if there are any outliers, clusters or other artefacts.

Let's try this new hammer on our data!


In [43]:
def get_phrase_embedding(phrase):
    """
    Convert phrase to a vector by aggregating it's word embeddings. See description above.
    """
    # 1. lowercase phrase
    phrase_low = phrase.lower()
    # 2. tokenize phrase
    phrase_tok = tokenizer.tokenize(phrase_low)
    vecs = [model.get_vector(t) for t in phrase_tok if t in model.key_to_index]

    if not vecs:
        # if all words are missing from vocabulary, return zeros
        return np.zeros((model.vector_size,), dtype='float32')

    # 3. average word vectors for all words in tokenized phrase
    return np.mean(np.vstack(vecs), axis=0).astype('float32')
    # vector = np.zeros([model.vector_size], dtype='float32')

In [44]:
vector = get_phrase_embedding("I'm very sure. This never happened to me before...")

assert np.allclose(vector[::10],
                   np.array([ 0.31807372, -0.02558171,  0.0933293 , -0.1002182 , -1.0278689 ,
                             -0.16621883,  0.05083408,  0.17989802,  1.3701859 ,  0.08655966],
                              dtype=np.float32))
assert np.array_equal(get_phrase_embedding("thisisgibberish"), np.zeros([model.vector_size], dtype='float32')), "corner case for all missing words should be handled as described in the function comments"

In [45]:
# let's only consider ~5k phrases for a first run.
chosen_phrases = data[::len(data) // 1000]

# compute vectors for chosen phrases
# phrase_vectors = # YOUR CODE
phrase_vectors = np.array([get_phrase_embedding(phrase) for phrase in chosen_phrases])
phrase_vectors

array([[ 0.04363502,  0.07049451,  0.02798353, ..., -0.02216815,
         0.14445603, -0.00921343],
       [ 0.12106456,  0.05835532,  0.18340029, ..., -0.19093764,
         0.2600007 , -0.03742344],
       [ 0.0841652 ,  0.05124575,  0.04520828, ..., -0.270041  ,
         0.09629561, -0.08329602],
       ...,
       [ 0.21569163,  0.04237552, -0.00400152, ..., -0.2054347 ,
        -0.04555578, -0.04466971],
       [ 0.1998261 ,  0.16787907,  0.05831072, ...,  0.05013187,
        -0.01774666, -0.23646466],
       [ 0.14187764,  0.07504999,  0.22541565, ...,  0.00450463,
         0.31108332,  0.00859667]], dtype=float32)

In [46]:
assert isinstance(phrase_vectors, np.ndarray) and np.isfinite(phrase_vectors).all()
assert phrase_vectors.shape == (len(chosen_phrases), model.vector_size)

In [47]:
# map vectors into 2d space with pca, tsne or your other method of choice
# don't forget to normalize

phrase_vectors_2d = TSNE().fit_transform(phrase_vectors)

phrase_vectors_2d = (phrase_vectors_2d - phrase_vectors_2d.mean(axis=0)) / phrase_vectors_2d.std(axis=0)

In [48]:
draw_vectors(phrase_vectors_2d[:, 0], phrase_vectors_2d[:, 1],
             phrase=[phrase[:50] for phrase in chosen_phrases],
             radius=20,)

figure(id='p1106', ...)

Finally, let's build a simple "similar question" engine with phrase embeddings we've built.

In [50]:
# compute vector embedding for all lines in data
data_vectors = np.array([get_phrase_embedding(l) for l in data])

In [51]:
phrase_vectors

array([[ 0.04363502,  0.07049451,  0.02798353, ..., -0.02216815,
         0.14445603, -0.00921343],
       [ 0.12106456,  0.05835532,  0.18340029, ..., -0.19093764,
         0.2600007 , -0.03742344],
       [ 0.0841652 ,  0.05124575,  0.04520828, ..., -0.270041  ,
         0.09629561, -0.08329602],
       ...,
       [ 0.21569163,  0.04237552, -0.00400152, ..., -0.2054347 ,
        -0.04555578, -0.04466971],
       [ 0.1998261 ,  0.16787907,  0.05831072, ...,  0.05013187,
        -0.01774666, -0.23646466],
       [ 0.14187764,  0.07504999,  0.22541565, ...,  0.00450463,
         0.31108332,  0.00859667]], dtype=float32)

In [52]:
def find_nearest(query, k=10):
    """
    given text line (query), return k most similar lines from data, sorted from most to least similar
    similarity should be measured as cosine between query and line embedding vectors
    hint: it's okay to use global variables: data and data_vectors. see also: np.argpartition, np.argsort
    """
    # YOUR CODE
    q = get_phrase_embedding(query)
    cosines = [vec @ q / np.linalg.norm(vec) / np.linalg.norm(q) for vec in phrase_vectors]

    indexes = np.argsort(cosines)[-k:] 

    result = [chosen_phrases[i] for i in indexes]
    
    return result

In [53]:
results = find_nearest(query="How do i enter the matrix?", k=10)

print(''.join(results))

assert len(results) == 10 and isinstance(results[0], str)
# assert results[0] == 'How do I get to the dark web?\n'
# assert results[3] == 'What can I do to save the world?\n'

What is the best way to read a fictional book? Do you take notes when you are reading? Do you read again these notes later?
How do you choose your first bank?
R2I - How did you plan R2I from US if you own the house, i mean job search, timeline etc ?
How do I find out if I have Siri on my phone?
How do I learn to enter journal entries online in 2 weeks or so?
How do I run a shell script from Java code?
My WhatsApp chat backup got deleted from Google, I need to switch from one Android to another, the chat is there only on the phone. What should I do?
How do I listen a song from you?
If I wanted to learn about the Roman Empire,what would be the best books to read?
How do I learn Calculus on my own?



In [54]:
find_nearest(query="How does Trump?", k=10)

['What is BusyBox used for?\n',
 'How do you feel when your question is unanswered on Quora?\n',
 'How can we say that climate change does not bring about health emergency?\n',
 'The education system is outdated. What would you do to change it?\n',
 'Why does India so scared of CPEC?\n',
 'What makes you sad about India?\n',
 'What were some things India did not do but takes credit for?\n',
 'What does it feel like to be an IITian?\n',
 'What might happen now that President-elect Donald Trump has won the election? What will be the impact?\n',
 'Does Donald Trump actually think he can become President?\n']

In [55]:
find_nearest(query="Why don't i ask a question myself?", k=10)

['Why should I ask my first question?\n',
 "Why do some people 'hate' drugs or people who ever use them? Isn't that a bit extreme?\n",
 "What should I do if someone doesn't reply to my email?\n",
 "Why do I feel like I'm not living my life?\n",
 "I need to gain weight, but I don't have abs. It's frustrating as heck. (150ibs 17 year old male) what should I do?\n",
 "I am 23 and don't know what I want. My life is very boring, I am depressed and frustrated, I don't have any good friends to share my feelings with. I don't even have a girlfriend. Sometimes I want to quit. What should I do?\n",
 "My ex bf says he doesn't have feelings for me right now. Why won't he just say I don't have feelings for you anymore?\n",
 "I'm really pretty but I don't want to be I hate the attention and dudes hitting on me what should I do?\n",
 "How do I get a girl's phone number in a library?\n",
 "What's a funny thing?\n"]

__Now what?__
* Try running TSNE on all data, not just 1000 phrases
* See what other embeddings are there in the model zoo: `gensim.downloader.info()`
* Take a look at [FastText](https://github.com/facebookresearch/fastText) embeddings
* Optimize `find_nearest` with locality-sensitive hashing: use [nearpy](https://github.com/pixelogik/NearPy) or `sklearn.neighbors`.